In [217]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [218]:
# Paths
DATA_PATH = os.path.join('..', '..', 'data', 'raw', 'go_emotions_dataset.csv')
OUTPUT_DIR = os.path.join('..', '..', 'data', 'processed')
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'go_emotions_cleaned.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('DATA_PATH ->', DATA_PATH)
print('OUTPUT_FILE ->', OUTPUT_FILE)

DATA_PATH -> ..\..\data\raw\go_emotions_dataset.csv
OUTPUT_FILE -> ..\..\data\processed\go_emotions_cleaned.csv


In [219]:
df = pd.read_csv(DATA_PATH)
print('Loaded rows, cols:', df.shape)

Loaded rows, cols: (211225, 31)


In [220]:
print('Columns sample:', df.columns[:10].tolist())

Columns sample: ['id', 'text', 'example_very_unclear', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion']


In [221]:
df.head(3)

,id,text,example_very_unclear,admiration,amusement,anger,annoyance,approval,caring,confusion,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,eew5j0j,That game hurt.,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,eemcysk,>sexuality shouldn’t be a grouping category I...,True,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ed2mah1,"You do right, if you don't care then fuck 'em!",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [222]:
# normalize the text (lowercase)
df['clean_text'] = df['text'].astype(str).str.lower()

In [223]:
df.head(3)

,id,text,example_very_unclear,admiration,amusement,anger,annoyance,approval,caring,confusion,...,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral,clean_text
0,eew5j0j,That game hurt.,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,that game hurt.
1,eemcysk,>sexuality shouldn’t be a grouping category I...,True,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,>sexuality shouldn’t be a grouping category i...
2,ed2mah1,"You do right, if you don't care then fuck 'em!",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,"you do right, if you don't care then fuck 'em!"


In [224]:
# Check for urls
url_pattern = r'http\S+|www\S+|https\S+'
urls_exist = df['clean_text'].str.contains(url_pattern, regex=True).sum()
print(f"Rows containing URLs: {urls_exist}")

Rows containing URLs: 16


In [225]:
# Replace urls with [URL]
df['clean_text'] = df['clean_text'].str.replace(url_pattern, '[URL]', regex=True)

In [226]:
url_check = df['clean_text'].str.contains(url_pattern, regex=True)

# Count how many rows still have URLs
print("Rows with URLs remaining:", url_check.sum())

Rows with URLs remaining: 0


In [227]:
# Check for emails
email_pattern = r'\b[\w.-]+@[\w.-]+\.\w{2,4}\b'
emails_exist = df['clean_text'].str.contains(email_pattern, regex=True).sum()
print(f"Rows containing emails: {emails_exist}")

Rows containing emails: 0


In [228]:
# Check for phone numbers
phone_pattern = r'\b\+?\d[\d\s\-()]{6,}\b'
phones_exist = df['clean_text'].str.contains(phone_pattern, regex=True).sum()
print(f"Rows containing phone numbers: {phones_exist}")

Rows containing phone numbers: 114


In [229]:
# Replace phone numbers with [PHONE]
df['clean_text'] = df['clean_text'].str.replace(phone_pattern, '[PHONE]', regex=True)

In [230]:
# Check if any phone numbers still exist in clean_text
phone_remaining = df['clean_text'].str.contains(phone_pattern, regex=True)

# Count how many rows still have phone numbers
print("Rows with phone numbers remaining:", phone_remaining.sum())

Rows with phone numbers remaining: 0


In [231]:
#Check for long numbers (4+ digits)
num_pattern = r'\b\d{4,}\b'
long_nums_exist = df['clean_text'].str.contains(num_pattern, regex=True).sum()
print(f"Rows containing long numbers: {long_nums_exist}")

Rows containing long numbers: 1581


In [232]:
long_nums = df['clean_text'].str.contains(num_pattern, regex=True)
df[long_nums]['clean_text'].sample(10)

72459     if his seat was vacant, would there be a speci...
9815                             oh come on mom..it's 2019.
3279      it may have been funny in 2010, but now it's c...
85151     sorry this and [name] shot in game 5 of the 19...
205148    this was a year where late breakers broke towa...
111238    looks like russian agitprop is pivoting to tul...
190484    so sorry for your loss! i lost my cat 6 years ...
203188    what? the claim is 2018 best shooters. not ove...
169469    yes you should, although it's only in 2300. su...
196367       i love [name]. i hope she can hold on in 2020.
Name: clean_text, dtype: object

In [233]:
df['clean_text'] = df['clean_text'].str.replace(num_pattern, '[NUM]', regex=True)

In [234]:
long_nums_remaining = df['clean_text'].str.contains(num_pattern, regex=True)

# Count how many rows still have long numbers
print("Rows with long numbers remaining:", long_nums_remaining.sum())

Rows with long numbers remaining: 0


In [235]:
# Keep letters, numbers, basic punctuation, brackets
df['clean_text'] = df['clean_text'].str.replace(r'[^a-zA-Z0-9\s\[\]\.\,\!\?\-]', '', regex=True)

# Normalize whitespace
df['clean_text'] = df['clean_text'].str.replace(r'\s+', ' ', regex=True).str.strip()


In [236]:
texts = df['clean_text'].astype(str)

In [237]:
# Check if emojis exist in the dataset
emoji_pattern = re.compile(
    "[\U0001F600-\U0001F64F]|"  # emoticons
    "[\U0001F300-\U0001F5FF]|"  # symbols & pictographs
    "[\U0001F680-\U0001F6FF]|"  # transport & map symbols
    "[\U0001F700-\U0001F77F]|"  # alchemical symbols
    "[\U0001F780-\U0001F7FF]|"  # Geometric Shapes Extended
    "[\U0001F800-\U0001F8FF]|"  # Supplemental Arrows-C
    "[\U0001F900-\U0001F9FF]|"  # Supplemental Symbols and Pictographs
    "[\U0001FA00-\U0001FA6F]|"  # Chess Symbols
    "[\U0001FA70-\U0001FAFF]|"  # Symbols and Pictographs Extended-A
    "[\U00002702-\U000027B0]|"  # Dingbats
    "[\U000024C2-\U0001F251]"   # Enclosed characters
    "+", flags=re.UNICODE
)

# Count rows containing emojis
emojis_exist = df['clean_text'].str.contains(emoji_pattern, regex=True).sum()
print(f"Rows containing emojis: {emojis_exist}")

Rows containing emojis: 0


In [238]:
mask_map = {
    # DIRECT IDENTIFIERS 

    # Usernames / handles
    r'\b@[A-Za-z0-9_]+\b': '[USERNAME]',
    r'\b[A-Za-z0-9._-]{3,}#[0-9]{4}\b': '[USERNAME]',  

    # Emails
    r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b': '[EMAIL]',

    # Phone numbers (international + local)
    r'\b(?:\+?\d{1,3}[\s-]?)?(?:\d{2,4}[\s-]?){2,4}\d\b': '[PHONE]',

    # URLs
    r'\bhttps?://\S+|\bwww\.\S+\b': '[URL]',

    # Government / institutional IDs
    r'\b(?:nic|passport|student\s*id|employee\s*id|patient\s*id)\b[\s:]*\w+': '[ID]',

    # IP addresses
    r'\b(?:\d{1,3}\.){3}\d{1,3}\b': '[IP]',


    # PRECISE LOCATION (MASK GRANULAR ONLY)

    # Street-level addresses
    r'\b\d{1,4}\s+[A-Za-z]+\s+(?:street|st|road|rd|lane|ln|avenue|ave|boulevard|blvd)\b': '[ADDRESS]',

    # Self-referenced precise areas
    r'\b(?:my|our)\s+(?:street|lane|road|block|neighborhood|area|building)\b': '[LOCATION]',

    # Named institutions (schools, hospitals, workplaces)
    r'\b(?:school|college|university|hospital|clinic|company|office|workplace)\s+[A-Z][A-Za-z& ]{2,}\b': '[ORG]',

    # Standalone organization mentions
    r'\b(?:at|from|in)\s+[A-Z][A-Za-z& ]{2,}\b': '[ORG]',

    # TEMPORAL IDENTIFIERS (EXACT ONLY)

    # Exact calendar dates
    r'\b(?:january|february|march|april|may|june|july|august|'
    r'september|october|november|december)\s+\d{1,2}'
    r'(?:st|nd|rd|th)?(?:,\s*\d{4})?\b': '[DATE]',

    # Clock times
    r'\b\d{1,2}:\d{2}\s?(?:am|pm)?\b': '[TIME]',

    # Explicit years
    r'\b(?:19|20)\d{2}\b': '[YEAR]',

    # Durations linked to events (reduces linkage risk)
    r'\b\d+\s+(?:days?|weeks?|months?|years?)\b': '[DURATION]',


    # RELATIONSHIPS (MASK IDENTITY, KEEP ROLE)

    r'\b(?:my|our)\s+(?:mother|father|mom|dad|sister|brother|wife|husband|partner)\s+[A-Z][a-z]+\b': '[RELATIONSHIP_NAME]',

    # FINANCIAL (OPTIONAL, SAFE)

    r'\$\d+(?:,\d{3})*(?:\.\d{2})?': '[MONEY]',
}

In [239]:
# Check which categories exist and how many times
for pattern, token in mask_map.items():
    count = texts.str.contains(pattern, flags=re.IGNORECASE, regex=True).sum()
    if count > 0:
        print(f"{token}: found in {count} rows")

[PHONE]: found in 40 rows
[ID]: found in 6 rows
[ADDRESS]: found in 9 rows
[LOCATION]: found in 65 rows
[ORG]: found in 1402 rows
[ORG]: found in 37648 rows
[DATE]: found in 79 rows
[YEAR]: found in 8 rows
[DURATION]: found in 2490 rows
[RELATIONSHIP_NAME]: found in 763 rows


In [240]:
import re

def apply_masks(text, mask_map):
    if not isinstance(text, str):
        return text

    for pattern, token in mask_map.items():
        text = re.sub(pattern, token, text, flags=re.IGNORECASE)

    # Cleanup spacing artifacts
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\s*([.,!?])', r'\1', text)

    return text

df['clean_text'] = df['clean_text'].apply(lambda x: apply_masks(x, mask_map))

In [241]:
# Count exact duplicate clean_text rows
duplicate_mask = df['clean_text'].duplicated(keep=False)  # keep=False marks all duplicates as True
num_duplicates = duplicate_mask.sum()
print(f"Number of rows that have exact duplicate clean_text: {num_duplicates}")

Number of rows that have exact duplicate clean_text: 211156


In [242]:
df = df.drop_duplicates(subset=['clean_text']).reset_index(drop=True)
print("Rows after removing exact duplicate clean_text:", df.shape[0])

Rows after removing exact duplicate clean_text: 57414


In [243]:
df['clean_text'] = df['clean_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [244]:
null_count = df['clean_text'].isna().sum()
print("Null values in clean_text:", null_count)

Null values in clean_text: 0


In [245]:
# Save dataset
df.to_csv(OUTPUT_FILE, index=False)

print(f"Cleaned dataset saved successfully at: {OUTPUT_FILE}")
print("Final dataset shape:", df.shape)

Cleaned dataset saved successfully at: ..\..\data\processed\go_emotions_cleaned.csv
Final dataset shape: (57414, 32)
